import sys
!{sys.executable} -m pip install pandas

In [ ]:
import sys
!{sys.executable} -m pip install pandas
import pandas

Reading the CSV files with Raw Results of Jaccard Index Based Focal Method Match

In [ ]:
import pandas as pd
import numpy as np

file_path='/mnt/efs/people/rabaisha/GitLab/change_aware_utg/bedrock_experiment/Results/Static_Analysis_and_Claude_comparison2.csv'

def read_csv(file_path):
    """Read CSV file into a pandas DataFrame"""
    df = pd.read_csv(file_path, sep=',', header=0)
    return df

df = read_csv(file_path)
#number of rows in df
df = df.drop('Focal_method_Class_name', axis=1)
df.count()

In [ ]:
df_no_jaccard=df[df['Static_Analysis_Result'].isna()]
df_no_jaccard.count()

In [ ]:
#Find rows where Jaccard Similarity Finds a Match
df_jaccard=df[df['Static_Analysis_Result'].notnull()]
#dfs = dfs.drop('Focal_method_Class_name', axis=1)
df_jaccard.count()

In [12]:
unique_method = df.groupby(['git_link','Static_Analysis_Result']).size()
unique_method.count()

795

In [ ]:
#panda shows df index?
#dfs = dfs.dropna()
df_jaccard.loc[1277]

In [ ]:
#Finding Match with Jaccard Result and Claude Result
df_jaccard['SA'] = df_jaccard['Static_Analysis_Result'].str.split('#')
df_jaccard['CL'] = df_jaccard['claude_result'].str.split('#')
df_jaccard = df_jaccard.reset_index()  # make sure indexes pair with number of rows
df_jaccard['isMatch'] = 0
count = 0
for index, row in df_jaccard.iterrows():
    if len(row['SA']) != 2:
        print('SA Error!!', index)
        continue

    if len(row['CL']) != 2:
        print('CL Error!!', index, row['CL'])
        count=count+1
        continue
    
    assert(len(row['SA'])==len((row['CL'])))

    if str(row['SA'][0]) == str(row['CL'][0]):
        #print(str(row['SA'][0]), row['CL'][0])
        df_jaccard.at[index, 'isMatch'] = 1
    
print(count)
    
    #assert(len(row['SA']==len((row['CL']))))

In [43]:
#dfs.loc[[2,1453]]
#dfs.loc[[2]]
utg_fm_jaccard=df_jaccard
utg_fm_jaccard_claude_confirmed=df_jaccard[df_jaccard['isMatch']==1]


In [ ]:
#Claude and Jaccard agrees with FM name but disagrees with arguments
dfs1=df_jaccard[(df_jaccard['isMatch']==1) & (df_jaccard['Findings']=='MisMatched')]
dfs1.count()

In [47]:
#Saving to csvs
from pathlib import Path  
dir_path='/home/ec2-user/rabaisha/GitLab/change_aware_utg/rabaisha_analysis/'

file_name = 'utg_fm_original.csv'
filepath = Path(dir_path+file_name)  
filepath.parent.mkdir(parents=True, exist_ok=True)  
df.to_csv(filepath, index=False) 

file_name = 'utg_fm_no_jaccard.csv'
filepath = Path(dir_path+file_name)  
filepath.parent.mkdir(parents=True, exist_ok=True)  
df_no_jaccard.to_csv(filepath, index=False) 

file_name = 'utg_fm_jaccard.csv'
filepath = Path(dir_path+file_name)  
filepath.parent.mkdir(parents=True, exist_ok=True)  
utg_fm_jaccard.to_csv(filepath, index=False) 

file_name = 'utg_fm_jaccard_claude_confirmed.csv'
filepath = Path(dir_path+file_name)  
filepath.parent.mkdir(parents=True, exist_ok=True)  
utg_fm_jaccard_claude_confirmed.to_csv(filepath, index=False) 
